# KV 캐시: 효율적인 자기회귀 추론
## KV 캐시로 LLM 추론 가속화하기

- **Tutorial:** KV 캐시: 효율적인 자기회귀 추론
- **Section:** KV 캐시로 추론 가속화

---

이 노트북은 LLM(대형 언어 모델)이 텍스트를 생성할 때 **왜 느려지는지**,  
그리고 **KV 캐시**라는 기법이 어떻게 그 문제를 해결하는지를  
numpy만으로 직접 구현하며 이해합니다.

수식보다 **코드 → 출력 → 직관** 순서로 읽어가세요.

## 이 노트북 읽는 법

### 학습 목표
1. **자기회귀 생성**이 왜 느린지 이해한다
2. **Q / K / V**가 무엇인지, 어떤 shape으로 만들어지는지 추적한다
3. **KV 캐시**가 어떻게 중복 계산을 없애는지 직접 구현한다
4. 캐시가 메모리를 얼마나 쓰는지 실제 모델 수치로 계산한다

### 읽는 순서
1. 각 코드 셀 위의 **Markdown 설명**을 먼저 읽으세요 (개념 파악)
2. 코드 안의 **주석**을 읽으세요 (구현 의도 파악)
3. **출력(print 결과)**을 보며 shape와 숫자의 변화를 확인하세요
4. 마지막 **실험 셀**에서 파라미터를 바꿔보며 결과를 직접 확인하세요

### 전제 지식
- Python 기초 (함수, 반복문, 딕셔너리)
- numpy 기초 (배열, shape, `@` 행렬 곱)
- Transformer/Attention에 대한 대략적인 개념 (몰라도 진행 가능합니다)

## 배경 개념 ① — 자기회귀(Autoregressive) 생성이란?

LLM은 텍스트를 **한 번에 전부 생성하지 않습니다**.  
**토큰(token) 하나씩, 순서대로** 생성합니다.

```
'오늘 날씨가' -> '좋다'  생성 과정:

  Step 0: 입력 ['오늘']           -> 다음 토큰 '날씨가' 예측
  Step 1: 입력 ['오늘', '날씨가'] -> 다음 토큰 '좋다'  예측
  Step 2: 입력 ['오늘', '날씨가', '좋다'] -> 다음 토큰 '.' 예측
  ...
```

매 스텝마다 **이전에 생성한 모든 토큰을 다시 봐야** 합니다.  
여기서 문제가 생깁니다:  
Attention 계산을 위해 **이전 토큰들의 K, V를 매 스텝 처음부터 다시 계산**하게 됩니다.

> 핵심 질문: K와 V는 이전 스텝과 달라지지 않는데, 왜 다시 계산할까요?

In [ ]:
import numpy as np

# numpy : 파이썬의 핵심 행렬 연산 라이브러리
# 딥러닝 프레임워크(PyTorch) 없이도 행렬 곱셈으로 Attention 원리를 직접 구현할 수 있습니다

print('=' * 60)
print('KV 캐시: 자기회귀 추론 가속화 실습')
print('=' * 60)

# -----------------------------------------------------------
# [Softmax 함수]
#
# 임의의 숫자 배열을 '확률 분포'로 바꿔주는 함수입니다.
# - 모든 값이 0 이상 1 이하
# - 모든 값의 합 = 1.0
#
# 예시:
#   입력  : [2.0,  1.0,  0.5]
#   출력  : [0.59, 0.24, 0.17]   (합계 = 1.00)
#
# Attention에서는 '어떤 토큰에 얼마나 집중할지'를 확률로 표현할 때 씁니다.
# -----------------------------------------------------------
def softmax(x, axis=-1):
    # np.max를 빼는 이유: 수치 안정성(overflow 방지)을 위한 트릭
    # 결과값은 빼지 않은 것과 수학적으로 동일합니다
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

# softmax 동작 확인
test_input  = np.array([2.0, 1.0, 0.5])
test_output = softmax(test_input)
print(f'\n[softmax 예시]')
print(f'  입력 : {test_input}')
print(f'  출력 : {test_output.round(4)}')
print(f'  합계 : {test_output.sum():.4f}  (항상 1.0이 됩니다)')

# -----------------------------------------------------------
# [하이퍼파라미터 설정]
#
# 모델의 '설계 변수'입니다. 학습 전에 사람이 결정합니다.
#
# d_model  : 하나의 토큰을 몇 개의 숫자(차원)로 표현하는가?
#            예) '나는' -> [0.3, -0.1, 0.8, ...(64개의 숫자)]
#
# n_heads  : Attention을 몇 개의 '관점'에서 동시에 볼 것인가?
#            예) 4개 헤드 -> 4가지 다른 관점에서 동시에 분석
#
# d_head   : 각 헤드 내부의 차원 수 = d_model / n_heads
#            예) 64 / 4 = 16
# -----------------------------------------------------------
d_model = 64                     # 토큰 임베딩 차원
n_heads = 4                      # Multi-Head Attention 헤드 수
d_head  = d_model // n_heads     # 헤드당 차원 (= 16)

print(f'\n[모델 설정]')
print(f'  d_model  = {d_model}  (토큰 1개를 {d_model}차원 벡터로 표현)')
print(f'  n_heads  = {n_heads}   (동시에 {n_heads}가지 관점으로 Attention 수행)')
print(f'  d_head   = {d_head}  (헤드 1개당 {d_head}차원 사용)')
print(f'  검증: n_heads x d_head = {n_heads} x {d_head} = {n_heads * d_head} == d_model ({d_model}) [OK]')

np.random.seed(42)   # 랜덤 시드 고정 -> 매번 같은 숫자 재현

# -----------------------------------------------------------
# [Q, K, V 투영(Projection) 행렬]
#
# Transformer에서 입력 임베딩 X를 세 가지 공간으로 변환합니다:
#
#   Q = X @ W_Q   ->  Query  : '내가 찾고 싶은 것이 무엇인가?'
#   K = X @ W_K   ->  Key    : '나는 어떤 정보를 갖고 있는가?'
#   V = X @ W_V   ->  Value  : '실제로 전달할 정보 내용'
#
# 도서관 검색 비유:
#   Q = 내 검색어 ('파이썬 관련 책')
#   K = 각 책의 제목/태그 ('파이썬 입문', '머신러닝', ...)
#   V = 책의 실제 내용 (빌려올 정보)
#
# 형태: (d_model, n_heads x d_head) = (64, 64)
#
# * 0.1을 곱하는 이유: 초기 가중치 크기가 너무 크면 학습이 불안정해집니다
# -----------------------------------------------------------
W_Q = np.random.randn(d_model, n_heads * d_head) * 0.1
W_K = np.random.randn(d_model, n_heads * d_head) * 0.1
W_V = np.random.randn(d_model, n_heads * d_head) * 0.1

print(f'\n[가중치 행렬 shape]')
print(f'  W_Q : {W_Q.shape}   입력({d_model}차원) -> Q 공간({n_heads * d_head}차원)')
print(f'  W_K : {W_K.shape}   입력({d_model}차원) -> K 공간({n_heads * d_head}차원)')
print(f'  W_V : {W_V.shape}   입력({d_model}차원) -> V 공간({n_heads * d_head}차원)')
print(f'\n  [OK] 설정 완료. 다음 셀에서 Attention 계산을 단계별로 따라갑니다.')

## 배경 개념 ② — Attention 메커니즘

KV 캐시를 이해하려면 Attention이 **어떤 순서로 계산되는지** 알아야 합니다.

```
입력 X : (seq_len, d_model)
    |
    |-- x W_Q -->  Q (Query)   shape: (seq_len, d_head)
    |-- x W_K -->  K (Key)     shape: (seq_len, d_head)
    +-- x W_V -->  V (Value)   shape: (seq_len, d_head)

Attention Score  = Q @ K.T / sqrt(d_head)   shape: (seq_len, seq_len)
Attention Weight = Softmax(Score)            shape: (seq_len, seq_len) <- 각 행의 합 = 1
Output           = Weight @ V               shape: (seq_len, d_head)
```

### 각 변수의 의미

| 변수 | 역할 | 비유 |
|------|------|------|
| **Q (Query)** | '내가 무엇에 집중해야 하나?' | 검색창의 검색어 |
| **K (Key)** | '나는 어떤 정보를 갖고 있나?' | 각 웹페이지의 제목/태그 |
| **V (Value)** | '실제로 전달할 내용' | 웹페이지의 본문 내용 |

### KV 캐시의 근거
- 새 토큰의 **Q는 매 스텝 새로 계산** 합니다
- 이전 토큰들의 **K, V는 스텝이 바뀌어도 값이 달라지지 않습니다** — 저장해두면 됩니다!

In [ ]:
# -----------------------------------------------------------
# Attention 계산을 4단계로 나눠 직접 추적합니다.
#
# 시나리오: '나는 / 학교에 / 갔다' 라는 3개의 토큰이 있다고 가정
# 각 토큰은 d_model=64 차원의 벡터로 표현됩니다.
# (실제로는 학습된 임베딩이지만, 여기서는 이해를 위해 랜덤 숫자를 사용)
# -----------------------------------------------------------
print('=' * 60)
print('단계별 Attention 계산 추적')
print('=' * 60)

token_names = ['나는', '학교에', '갔다']
seq_len_ex  = len(token_names)   # = 3

# 입력 시퀀스 X : shape = (3, 64)
# 3개 토큰 각각을 64차원 벡터로 표현한 행렬
X = np.random.randn(seq_len_ex, d_model) * 0.1

print(f'\n[입력 시퀀스 X]')
print(f'  shape: {X.shape}  <- {seq_len_ex}개 토큰 x {d_model}차원')
for i, name in enumerate(token_names):
    print(f'  X[{i}] {name!r:^8}: [{X[i,0]:.3f}, {X[i,1]:.3f}, {X[i,2]:.3f}, ... (총 {d_model}개 숫자)]')

# -----------------------------------------------------------
# [Step 1] Q, K, V 계산
#
# 행렬 곱:  X (3, 64) @ W_Q (64, 64) = Q (3, 64)
# 각 행 = 해당 토큰의 Q (또는 K, V) 벡터
# -----------------------------------------------------------
Q = X @ W_Q   # shape: (3, 64)
K = X @ W_K   # shape: (3, 64)
V = X @ W_V   # shape: (3, 64)

sep = '-' * 50
print(f'\n{sep}')
print('[Step 1] Q, K, V 투영 계산')
print(sep)
print(f'  X  shape : {X.shape}')
print(f'  W_Q shape: {W_Q.shape}')
print(f'  Q = X @ W_Q -> {Q.shape}   (각 행 = 해당 토큰의 Query 벡터)')
print(f'  K = X @ W_K -> {K.shape}   (각 행 = 해당 토큰의 Key 벡터)')
print(f'  V = X @ W_V -> {V.shape}   (각 행 = 해당 토큰의 Value 벡터)')

# -----------------------------------------------------------
# [Step 2] Attention Score 계산
#
# score[i][j] = i번 토큰의 Query . j번 토큰의 Key  (내적)
#             = '토큰 i는 토큰 j에 얼마나 집중해야 하나?'
#
# Q @ K.T : (3, 64) @ (64, 3) = (3, 3)
#            -> 3x3 '관심도 행렬'
#
# / sqrt(d_head) : 차원이 클수록 내적값이 커지므로 나눠서 정규화
#   이 값이 너무 크면 softmax 이후 gradient가 사라지는 문제가 생깁니다
# -----------------------------------------------------------
scale  = np.sqrt(d_head)    # = sqrt(16) = 4.0
scores = Q @ K.T / scale    # shape: (3, 3)

print(f'\n{sep}')
print('[Step 2] Attention Score = Q @ K.T / sqrt(d_head)')
print(sep)
print(f'  Q @ K.T : {Q.shape} @ {K.T.shape} = {scores.shape}')
print(f'  스케일  : / sqrt({d_head}) = / {scale:.2f}')
print(f'\n  Score 행렬 (행=Query 토큰, 열=Key 토큰):')
header = f'  {"":^8}'
for name in token_names:
    header += f'  {name:^8}'
print(header)
for i, from_tok in enumerate(token_names):
    row = f'  Q:{from_tok:<6}'
    for j in range(seq_len_ex):
        row += f'  {scores[i,j]:+.4f}'
    print(row)
print(f'\n  예: score[0][2] = {scores[0,2]:+.4f}')
print(f'      -> 나는(Query)이 갔다(Key)에 집중해야 할 원시 점수')

# -----------------------------------------------------------
# [Step 3] Softmax -> Attention Weight
#
# 점수(score)를 확률(0~1)로 변환합니다.
# - 각 행의 합 = 1.0
# - 높은 점수 = 더 많이 집중 / 낮은 점수 = 덜 집중
# -----------------------------------------------------------
weights = softmax(scores, axis=-1)   # shape: (3, 3)

print(f'\n{sep}')
print('[Step 3] Softmax(Score) -> Attention Weight')
print(sep)
print(f'  shape: {weights.shape}  (각 행의 합 = 1.0)')
print(f'\n  Weight 행렬 (확률분포):')
header2 = f'  {"":^8}'
for name in token_names:
    header2 += f'  {name:^8}'
header2 += f'  {"합계":^6}'
print(header2)
for i, from_tok in enumerate(token_names):
    row = f'  Q:{from_tok:<6}'
    for j in range(seq_len_ex):
        row += f'  {weights[i,j]:.4f}  '
    row += f'  {weights[i].sum():.4f} <- 항상 1.0'
    print(row)

# -----------------------------------------------------------
# [Step 4] Attention Output
#
# 각 토큰은 Attention Weight에 따라 V를 가중합산합니다.
#
# output = weights @ V
# shape: (3, 3) @ (3, 64) = (3, 64)
#
# 예: output[0] = weights[0,0]*V[0] + weights[0,1]*V[1] + weights[0,2]*V[2]
#     -> '나는'은 weight에 따라 모든 토큰의 V를 섞어서 가져옵니다
# -----------------------------------------------------------
output = weights @ V   # shape: (3, 64)

print(f'\n{sep}')
print('[Step 4] Output = Attention Weight @ V')
print(sep)
print(f'  weights @ V : {weights.shape} @ {V.shape} = {output.shape}')
print(f'  output[0] (나는) 앞 5개 값: {output[0,:5].round(4)}')
print(f'  output[1] (학교에) 앞 5개 값: {output[1,:5].round(4)}')
print(f'\n  입력 X와 output의 shape가 동일: {X.shape} -> {output.shape}')
print(f'  토큰 수와 차원이 유지되면서, 각 토큰은 다른 토큰의 정보를 흡수했습니다.')

## 문제: 자기회귀 생성에서 무슨 일이 일어나는가?

토큰이 늘어날수록 K, V 계산이 어떻게 중복되는지 살펴봅니다.

```
Step 1: ['오늘']                  K=[K0],       V=[V0]       계산
Step 2: ['오늘', '날씨가']        K=[K0, K1],   V=[V0, V1]  계산  <- K0, V0 중복!
Step 3: ['오늘', '날씨가', '좋다'] K=[K0,K1,K2], V=[V0,V1,V2]    <- K0,K1,V0,V1 중복!
```

토큰이 N개가 되면 K, V 계산 횟수:
- **캐시 없음**: 1 + 2 + 3 + ... + N = N(N+1)/2 번 계산  (이차 성장)
- **캐시 있음**: 1 + 1 + 1 + ... + 1 = N 번 계산  (선형 성장)

> 이미 계산한 K와 V는 다음 스텝에도 동일합니다!  
> 저장해두면 되는 것을 왜 매번 다시 계산할까요?

In [ ]:
# -----------------------------------------------------------
# 캐시 없이 자기회귀 생성을 시뮬레이션합니다.
# 매 스텝마다 처음부터 K, V를 다시 계산하는 방식을 직접 확인합니다.
# -----------------------------------------------------------
print('=' * 65)
print('캐시 없이 자기회귀 생성 -- 낭비가 얼마나 일어나는지 관찰')
print('=' * 65)

# 최대 5개 토큰을 순차적으로 생성한다고 가정합니다.
max_seq_len = 5

# 5개 토큰의 임베딩을 미리 준비합니다.
# 실제 생성 시에는 스텝마다 새 토큰이 나오지만,
# 여기서는 전체 임베딩을 미리 만들고 하나씩 꺼내 씁니다.
full_embeddings = np.random.randn(max_seq_len, d_model)

print(f'\n[시뮬레이션 설정]')
print(f'  최대 생성 토큰 수 : {max_seq_len}개')
print(f'  d_model          : {d_model}')
print(f'  n_heads * d_head : {n_heads * d_head}')

print(f'\n  Step | 현재 길이 | K,V 계산 행수 | 중복 재계산 | 이 스텝 FLOPS')
print('  ' + '-' * 62)

total_no_cache_flops = 0
no_cache_flops_per_step = []

for t in range(1, max_seq_len + 1):
    # 캐시 없이: 현재까지의 모든 토큰(t개)에 대해 K, V를 처음부터 재계산
    current_X = full_embeddings[:t]        # shape: (t, d_model)

    # t개 토큰 전부 K, V 계산 (핵심 낭비 지점!)
    K_all = current_X @ W_K               # shape: (t, n_heads*d_head)
    V_all = current_X @ W_V               # shape: (t, n_heads*d_head)

    # Q는 새 토큰(마지막 토큰) 1개에 대해서만 계산하면 됩니다
    Q_new = full_embeddings[t-1:t] @ W_Q  # shape: (1, n_heads*d_head)

    # 이 스텝의 FLOPS 추정
    # KV 재계산: t개 토큰 x d_model x d_kv x 2(K와 V)
    flops_kv   = t * d_model * (n_heads * d_head) * 2
    # Attention Score: Q(1xd_k) @ K^T(d_kxt) = (1xt)
    flops_attn = 1 * t * (n_heads * d_head)
    step_flops = flops_kv + flops_attn
    total_no_cache_flops += step_flops
    no_cache_flops_per_step.append(step_flops)

    redundant = t - 1  # 이미 이전에 계산했는데 다시 계산한 K,V 행 수

    print(f'  t={t:2d} | 토큰 {t:2d}개   | {t:2d}행 계산        |'
          f' {redundant:2d}행 낭비   | {step_flops:>10,}')
    if redundant > 0:
        print(f'       |           | (중 {redundant}행은 이전 스텝에서 이미 계산한 것!)  |')

print('  ' + '-' * 62)
print(f'  총계                                          {total_no_cache_flops:>10,}')
print(f'\n  ** 시퀀스가 길어질수록 중복 계산이 급격히 늘어납니다!')
print(f'     t=1: 새 계산 1개, 낭비 0개')
print(f'     t=5: 새 계산 1개, 낭비 4개')
print(f'     전체 K,V 계산의 대부분이 이미 한 계산의 반복입니다.')

## 해결책: KV 캐시 (Key-Value Cache)

**아이디어**: 한 번 계산한 K, V는 메모리에 저장해두고, 다음 스텝에서 꺼내 씁니다.

```
Step 1: 토큰0 처리 -> K0, V0 계산 -> 캐시 저장: [K0]
Step 2: 토큰1 처리 -> K1, V1 계산 -> 캐시 저장: [K0, K1]
                      K0는 캐시에서 꺼냄 (재계산 없음!)
Step 3: 토큰2 처리 -> K2, V2 계산 -> 캐시 저장: [K0, K1, K2]
                      K0, K1은 캐시에서 꺼냄 (재계산 없음!)
```

### 무엇을 새로 계산하고, 무엇을 재사용하나?

| 변수 | 매 스텝 행동 | 이유 |
|------|------------|------|
| **Q** (Query) | 새 토큰에 대해서만 계산 | 새 토큰의 Query만 필요 |
| **K** (Key) | 새 토큰 1개만 계산 후 캐시에 추가 | 이전 K는 값이 변하지 않음 |
| **V** (Value) | 새 토큰 1개만 계산 후 캐시에 추가 | 이전 V는 값이 변하지 않음 |

### 계산량 비교

| 방식 | 토큰 t에서 K,V *투영(projection)* 계산 횟수 | 총 N 스텝 |
|------|------------------------|----------|
| 캐시 없음 | t번 (전체 재계산) | N(N+1)/2 번 |
| **캐시 사용** | **1번 (새 토큰만)** | **N번** |

K, V를 새로 만드는 **투영 계산만 놓고 보면** 시퀀스 길이 1000에서 약 500배 차이가 납니다.

### 주의: 그런데 '전체' 속도 향상은 이보다 작습니다

Attention에는 K, V 투영 말고도 한 단계가 더 있습니다 — **Score 계산(Q·Kᵀ)** 입니다.  
이 단계는 캐시를 쓰든 안 쓰든, 새 토큰이 **지금까지의 모든 토큰**과 내적을 해야 하므로  
매 스텝 시퀀스 길이 t에 비례한 계산이 **그대로 남습니다**.

```
          K,V 투영 계산        Score 계산(Q·K^T)
캐시 없음:  t에 비례 (재계산)    t에 비례 (캐시 여부 무관)
캐시 사용:  상수 (새 토큰 1개)   t에 비례 (캐시 여부 무관) <- 그대로!
```

즉 **KV 캐시가 없애주는 부분은 K,V 투영 계산뿐**이고, Score 계산은 여전히 시퀀스 길이에 비례해서 커집니다.  
그래서 실제 '전체 FLOPS' 기준 속도 향상은 K,V 계산 횟수 비율(N배)보다 작게 나오고,  
시퀀스가 아주 길어지면 어떤 값 근처로 수렴하는 경향을 보입니다.

> 정확한 배율은 다음 코드 셀에서 직접 측정해 봅시다 — 추측 대신 숫자로 확인하는 것이 정확합니다.

In [ ]:
# -----------------------------------------------------------
# KV 캐시를 직접 구현하고, 각 스텝마다 캐시가 어떻게 자라는지 관찰합니다.
# -----------------------------------------------------------
print('=' * 60)
print('KV 캐시 구현 -- 한 번 계산한 K, V는 저장해서 재사용')
print('=' * 60)

# -----------------------------------------------------------
# [캐시 초기화]
#
# 캐시를 딕셔너리로 관리합니다:
#   cache['K'] : 지금까지 계산한 모든 토큰의 K를 쌓은 행렬
#   cache['V'] : 지금까지 계산한 모든 토큰의 V를 쌓은 행렬
#
# 처음에는 아무것도 없으므로 None으로 초기화합니다.
# -----------------------------------------------------------
def create_kv_cache():
    # 빈 KV 캐시를 만들어 반환합니다
    # 반환값: {'K': None, 'V': None}
    return {'K': None, 'V': None}


def update_cache(cache, new_K, new_V):
    # 새 토큰의 K, V를 캐시에 추가합니다.
    #
    # 입력:
    #   cache  : 현재 캐시 딕셔너리
    #   new_K  : 새 토큰의 Key   shape = (1, n_heads*d_head)
    #   new_V  : 새 토큰의 Value shape = (1, n_heads*d_head)
    #
    # 동작 예시:
    #   캐시 K가 (3, 64)이고 new_K가 (1, 64)이면
    #   -> np.vstack([(3,64), (1,64)]) = (4, 64)  (행 1개 추가)
    if cache['K'] is None:
        # 첫 토큰: 캐시에 없으므로 그냥 저장합니다
        cache['K'] = new_K   # shape: (1, n_heads*d_head)
        cache['V'] = new_V   # shape: (1, n_heads*d_head)
    else:
        # 이후 토큰: 기존 캐시 아래에 새 행을 이어 붙입니다
        # np.vstack = vertical stack (수직으로 쌓기, 행 추가)
        cache['K'] = np.vstack([cache['K'], new_K])  # (t-1, d_kv) + (1, d_kv) -> (t, d_kv)
        cache['V'] = np.vstack([cache['V'], new_V])
    return cache


def cached_attention_step(x_new, cache, W_Q, W_K, W_V):
    # 새 토큰 하나에 대해 KV 캐시를 활용한 Attention을 수행합니다.
    #
    # 핵심 차이:
    #   이전 방식 : 모든 t개 토큰에 대해 Q, K, V 계산
    #   캐시 방식 : 새 토큰 1개에 대해서만 Q, K, V 계산
    #              이전 토큰의 K, V는 캐시에서 꺼냄
    #
    # 입력:
    #   x_new  : 새 토큰 임베딩  shape = (1, d_model)
    #   cache  : 이전 K, V가 담긴 캐시
    # 반환:
    #   attn_output  : shape = (1, n_heads*d_head)
    #   cache        : 업데이트된 캐시
    #   attn_weights : shape = (1, t) -- 시각화용

    # 1단계: 새 토큰 1개에 대해서만 Q, K, V 계산
    Q_new = x_new @ W_Q   # shape: (1, 64)  <- 1개만!
    K_new = x_new @ W_K   # shape: (1, 64)  <- 1개만!
    V_new = x_new @ W_V   # shape: (1, 64)  <- 1개만!

    # 2단계: 캐시에 새 K, V 추가
    #   cache['K'] = [K_이전들..., K_new]
    cache = update_cache(cache, K_new, V_new)

    # 3단계: Attention Score 계산
    #   Q_new: (1, 64)  cache['K']: (t, 64)
    #   Q_new @ cache['K'].T = (1, 64) @ (64, t) = (1, t)
    #   -> 새 토큰이 이전 모든 토큰에 얼마나 집중하는가
    scale  = np.sqrt(d_head)
    scores = Q_new @ cache['K'].T / scale   # shape: (1, t)

    # 4단계: Softmax -> 확률
    attn_weights = softmax(scores, axis=-1)  # shape: (1, t)

    # 5단계: Value 가중합산
    #   (1, t) @ (t, 64) = (1, 64)
    attn_output = attn_weights @ cache['V']  # shape: (1, 64)

    return attn_output, cache, attn_weights


# -----------------------------------------------------------
# 실제 시뮬레이션: 토큰 5개를 순차적으로 처리
# -----------------------------------------------------------
cache = create_kv_cache()

print(f'\n[KV 캐시 스텝별 관찰]')
print('  Step | 새로 계산         | 캐시에서 재사용        | 캐시 K shape | Attention 크기')
print('  ' + '-' * 72)

total_cache_flops = 0
cache_flops_per_step = []

for t in range(max_seq_len):
    x_new = full_embeddings[t:t+1]   # 이번 스텝의 새 토큰, shape: (1, 64)

    attn_output, cache, weights = cached_attention_step(
        x_new, cache, W_Q, W_K, W_V
    )

    # 이 스텝의 FLOPS 계산
    flops_kv_new = 1 * d_model * (n_heads * d_head) * 2   # 새 토큰 K,V 1개만
    flops_attn   = 1 * (t + 1) * (n_heads * d_head)        # Q . K^T: (1, t+1)
    step_flops   = flops_kv_new + flops_attn
    total_cache_flops += step_flops
    cache_flops_per_step.append(step_flops)

    reused    = f'K,V {t}행' if t > 0 else '없음 (첫 토큰)'
    attn_size = f'(1, {t+1})'

    print(f'  t={t+1:2d} | K,V 1행 (새 토큰)  | {reused:^22} | '
          f'{str(cache["K"].shape):^12} | {attn_size:^14}')

print('  ' + '-' * 72)
print(f'  총 FLOPS (캐시 사용): {total_cache_flops:,}')

# -----------------------------------------------------------
# 스텝별 FLOPS 비교
# -----------------------------------------------------------
print(f'\n[스텝별 FLOPS 비교]')
print(f'  Step | 캐시 없음 FLOPS | 캐시 사용 FLOPS | 절약 비율')
print('  ' + '-' * 52)
for i in range(max_seq_len):
    nc = no_cache_flops_per_step[i]
    wc = cache_flops_per_step[i]
    print(f'  t={i+1:2d} | {nc:>13,}   | {wc:>13,}   | {nc/wc:.2f}x')
print('  ' + '-' * 52)
print(f'  합계 | {total_no_cache_flops:>13,}   | {total_cache_flops:>13,}   '
      f'| {total_no_cache_flops/total_cache_flops:.2f}x')
print(f'\n  [OK] 캐시 사용 시 {total_no_cache_flops/total_cache_flops:.2f}배 연산량 감소 (5 토큰 기준)')

In [ ]:
# -----------------------------------------------------------
# 다양한 시퀀스 길이에서 캐시 없음 vs 캐시 사용 FLOPS를 비교합니다.
# 실제 LLM은 수백~수천 토큰을 생성하므로 이 비교가 매우 중요합니다.
# -----------------------------------------------------------
print('=' * 65)
print('FLOPS 비교: 시퀀스 길이별 캐시 효과')
print('=' * 65)

print()
print('[FLOPS 공식 요약]')
print()
print('  캐시 없음 (t번째 스텝):')
print('    KV 투영 : t x d_model x d_kv x 2    <- t개 토큰 모두 재계산 (캐시로 없앨 수 있음)')
print('    Score   : 1 x t x d_kv              <- Q.K^T, 캐시 여부와 무관하게 항상 필요')
print()
print('  캐시 사용 (t번째 스텝):')
print('    KV 투영 : 1 x d_model x d_kv x 2    <- 새 토큰 1개만 계산 (캐시 효과!)')
print('    Score   : 1 x t x d_kv              <- 이 부분은 캐시를 써도 그대로 남음')
print()
print('  => KV 캐시는 "KV 투영" 부분만 없애줍니다.')
print('     Score 계산은 attention의 본질상 시퀀스 길이에 비례해 항상 필요하므로,')
print('     아래 표의 속도 향상 배율은 "KV 계산 횟수 비율(N배)"보다 작게 나타납니다.')
print()

d_kv = n_heads * d_head   # = 64

print(f'  {"시퀀스":^8} | {"캐시 없음 FLOPS":^20} | {"캐시 사용 FLOPS":^20} | {"속도 향상":^10}')
print('  ' + '-' * 68)

speedup_results = {}   # 시퀀스 길이별 실제 측정값을 저장해서 아래 설명에 그대로 사용

for seq_len in [10, 50, 100, 500, 1000, 5000]:
    no_cache_total   = 0
    with_cache_total = 0

    for t in range(1, seq_len + 1):
        # 캐시 없음: t개 토큰 전부 KV 재계산 + Score 계산
        no_cache_total   += t * d_model * d_kv * 2 + 1 * t * d_kv
        # 캐시 사용: 새 토큰 1개만 KV 계산 + Score 계산(동일하게 필요)
        with_cache_total += 1 * d_model * d_kv * 2 + 1 * t * d_kv

    speedup = no_cache_total / with_cache_total
    speedup_results[seq_len] = speedup
    # 간단한 막대 차트로 시각화
    bar = '#' * min(30, int(speedup / 5))

    print(f'  {seq_len:>6}개  | {no_cache_total:>18,}   | {with_cache_total:>18,}   |'
          f' {speedup:>6.1f}x  {bar}')

print(f'\n  실측 결과로 확인한 패턴:')
print(f'    {10:>5}개 토큰 -> {speedup_results[10]:>6.1f}배')
print(f'    {1000:>5}개 토큰 -> {speedup_results[1000]:>6.1f}배')
print(f'    {5000:>5}개 토큰 -> {speedup_results[5000]:>6.1f}배')
print(f'\n  처음에는 배율이 빠르게 증가하지만, 시퀀스가 매우 길어질수록')
print(f'  증가 속도가 둔화되며 어떤 값 근처로 수렴합니다.')
print(f'  (이유: Score 계산은 캐시로도 줄일 수 없는 O(t) 비용이기 때문입니다)')
print(f'\n  그래도 실무에서는 이 정도 배율만으로도 충분히 큰 효과입니다.')
print(f'  그래서 모든 실제 LLM 추론 엔진(vLLM, TGI, TensorRT-LLM)은')
print(f'  KV 캐시를 기본으로 사용합니다.')
print(f'\n  참고: 캐시를 써도 여전히 남는 Score 계산의 O(N^2) 비용을 더 줄이는 기법으로는')
print(f'  FlashAttention, PagedAttention 같은 추가 최적화가 있습니다 (다음 튜토리얼에서 다룸).')

## KV 캐시의 단점: 메모리 비용

KV 캐시는 속도를 높이지만 **메모리를 많이 씁니다**.

### KV 캐시 메모리 공식

```
총 메모리 = 2 (K + V)
          x n_layers    (레이어 수 -- 각 레이어마다 독립 캐시 필요)
          x seq_len     (현재 시퀀스 길이)
          x n_kv_heads  (KV 헤드 수)
          x d_head      (헤드당 차원)
          x bytes/값    (FP16=2, INT8=1, INT4=0.5)
```

### Attention 헤드 공유 방식 (메모리 절약 전략)

| 방식 | 설명 | KV 헤드 수 | 대표 모델 |
|------|------|-----------|----------|
| **MHA** | 모든 헤드가 독립 K, V 보유 | Q 헤드와 동일 | GPT-2, BERT |
| **GQA** | 여러 Q 헤드가 K, V 하나를 공유 | Q 헤드의 1/4~1/8 | LLaMA-2, Mistral |
| **MQA** | 전체 K, V가 단 1개 | 1 | Falcon |

GQA/MQA는 메모리를 크게 절약하지만, 품질이 약간 떨어질 수 있습니다.  
최신 모델 대부분은 GQA를 채택하여 품질과 메모리를 절충합니다.

In [ ]:
# -----------------------------------------------------------
# 실제 LLaMA-7B 파라미터로 KV 캐시 메모리를 계산합니다.
# -----------------------------------------------------------
print('=' * 60)
print('KV 캐시 메모리 분석 -- LLaMA-7B 기준')
print('=' * 60)

# LLaMA-7B 실제 아키텍처 파라미터
n_layers  = 32     # Transformer 레이어 수
n_heads   = 32     # Query 헤드 수
d_head    = 128    # 헤드당 차원
           # 참고: d_model = n_heads x d_head = 32 x 128 = 4096
max_seq   = 4096   # LLaMA-7B의 최대 컨텍스트 길이

print(f'\n[LLaMA-7B 아키텍처 파라미터]')
print(f'  레이어 수 (n_layers)  : {n_layers}')
print(f'  Query 헤드 수 (n_heads): {n_heads}')
print(f'  헤드당 차원 (d_head)   : {d_head}')
print(f'  d_model (= n_heads x d_head): {n_heads * d_head}')
print(f'  최대 시퀀스 길이       : {max_seq:,}')

# -----------------------------------------------------------
# [MHA / GQA / MQA 비교]
#
# 공식:
#   size_bytes = 2 x n_layers x seq_len x n_kv_heads x d_head x bytes_per_elem
#
#   2     : K 행렬 + V 행렬
#   FP16  : 값 하나당 2바이트 (16비트 부동소수점)
# -----------------------------------------------------------
print(f'\n[Attention 방식별 KV 캐시 크기] (FP16, seq_len={max_seq:,})')
print('  ' + '-' * 66)
print(f'  {"방식":^8} | {"KV 헤드":^8} | {"캐시 크기":^12} | {"설명":^28}')
print('  ' + '-' * 66)

# (방식 이름, KV 헤드 수, 설명)
attention_variants = [
    ('MHA',   32, 'Q 헤드마다 독립적인 K, V 보유'),
    ('GQA-8',  8, '4개 Q 헤드가 K, V 1개 공유'),
    ('GQA-4',  4, '8개 Q 헤드가 K, V 1개 공유'),
    ('MQA',    1, '전체 K, V 단 하나 공유'),
]

mha_size = None
for method, n_kv, desc in attention_variants:
    # 공식: 2(K+V) x 레이어 x 시퀀스 x KV헤드 x 헤드차원 x 2바이트(FP16)
    size_bytes = 2 * n_layers * max_seq * n_kv * d_head * 2
    size_gb    = size_bytes / 1e9
    if mha_size is None:
        mha_size = size_gb
        save_str = '기준'
    else:
        save_str = f'MHA 대비 {mha_size/size_gb:.0f}x 절약'
    print(f'  {method:^8} | {n_kv:^8} | {size_gb:^10.2f} GB | {desc:<28}  ({save_str})')

# -----------------------------------------------------------
# [양자화(Quantization) 효과]
#
# K, V 값을 낮은 정밀도(적은 비트)로 저장하면 메모리를 절약할 수 있습니다.
#
# FP32  : 4바이트 (일반 계산)
# FP16  : 2바이트 (추론 시 표준)
# INT8  : 1바이트 (양자화, 메모리 절반)
# INT4  : 0.5바이트 (적극적 양자화, 메모리 1/4)
#
# 단점: 비트 수가 낮아질수록 정밀도 손실 -> 품질 저하 가능성
# -----------------------------------------------------------
print(f'\n[양자화 방식별 KV 캐시 크기] (MHA 기준, seq_len={max_seq:,})')
print('  ' + '-' * 58)
print(f'  {"정밀도":^8} | {"바이트/값":^10} | {"캐시 크기":^12} | {"FP16 대비 절약":^14}')
print('  ' + '-' * 58)

fp16_size = 2 * n_layers * max_seq * n_heads * d_head * 2.0

for prec, bpp in [('FP32', 4.0), ('FP16', 2.0), ('INT8', 1.0), ('INT4', 0.5)]:
    size_bytes = 2 * n_layers * max_seq * n_heads * d_head * bpp
    size_gb    = size_bytes / 1e9
    ratio      = fp16_size / size_bytes

    if prec == 'FP16':
        ratio_str = '기준'
    elif prec == 'FP32':
        ratio_str = '2x 더 사용'
    else:
        ratio_str = f'{ratio:.0f}x 절약'

    print(f'  {prec:^8} | {bpp:^10} | {size_gb:^10.2f} GB | {ratio_str:^14}')

print(f'\n  실제 활용 팁:')
print(f'    GQA-8 + INT8 조합: MHA FP16 대비 메모리 약 8배 절약 가능')
print(f'    긴 컨텍스트(128K 토큰) 모델에서는 KV 캐시가 모델 파라미터보다 더 클 수 있습니다.')
print(f'    그래서 최신 모델(LLaMA-2, Mistral 등)은 GQA + 양자화를 기본으로 채택합니다.')

In [ ]:
# -----------------------------------------------------------
# 실험: 파라미터를 바꿔서 결과를 직접 확인해보세요!
# 아래 EXP_ 로 시작하는 변수들을 자유롭게 수정하고 실행해보세요.
# -----------------------------------------------------------
print('=' * 65)
print('실험: 파라미터를 바꿔 KV 캐시 효과를 직접 측정해보세요')
print('=' * 65)

# =====================================================
#  아래 값들을 자유롭게 바꿔보세요!
# =====================================================
EXP_seq_len     = 100      # 시퀀스 길이      <- 10, 100, 500, 1000, 10000
EXP_d_model     = 64       # 임베딩 차원      <- 64, 256, 512, 1024
EXP_n_heads     = 4        # 헤드 수          <- 2, 4, 8, 16, 32
EXP_n_layers    = 12       # 레이어 수        <- GPT-2: 12, LLaMA-7B: 32, GPT-4 추정: 96
EXP_max_context = 4096     # 최대 컨텍스트    <- 512, 2048, 8192, 32768, 128000
EXP_n_kv_heads  = 4        # KV 헤드 수       <- EXP_n_heads(MHA), 8(GQA), 1(MQA)
EXP_bits        = 2.0      # 정밀도(바이트/값) <- 4.0(FP32), 2.0(FP16), 1.0(INT8), 0.5(INT4)
# =====================================================

EXP_d_head = EXP_d_model // EXP_n_heads
EXP_d_kv   = EXP_n_heads * EXP_d_head  # = EXP_d_model

# 방식 이름 자동 결정
if EXP_n_kv_heads == EXP_n_heads:
    exp_attn_mode = 'MHA'
elif EXP_n_kv_heads == 1:
    exp_attn_mode = 'MQA'
else:
    exp_attn_mode = f'GQA (Q/KV={EXP_n_heads}/{EXP_n_kv_heads})'

bits_name = {4.0: 'FP32', 2.0: 'FP16', 1.0: 'INT8', 0.5: 'INT4'}.get(EXP_bits, f'{EXP_bits}B')

print(f'\n[실험 파라미터]')
print(f'  시퀀스 길이   : {EXP_seq_len:,} 토큰')
print(f'  d_model      : {EXP_d_model} (헤드 {EXP_n_heads}개 x {EXP_d_head}차원)')
print(f'  레이어 수     : {EXP_n_layers}')
print(f'  최대 컨텍스트 : {EXP_max_context:,} 토큰')
print(f'  Attention 방식: {exp_attn_mode}')
print(f'  정밀도        : {bits_name} ({EXP_bits}바이트/값)')

# FLOPS 비교
no_cache_total   = 0
with_cache_total = 0
for t in range(1, EXP_seq_len + 1):
    no_cache_total   += t * EXP_d_model * EXP_d_kv * 2 + 1 * t * EXP_d_kv
    with_cache_total += 1 * EXP_d_model * EXP_d_kv * 2 + 1 * t * EXP_d_kv

speedup = no_cache_total / with_cache_total

sep2 = '-' * 50
print(f'\n{sep2}')
print(f'[FLOPS 비교 -- {EXP_seq_len}개 토큰 생성]')
print(sep2)
print(f'  캐시 없음 : {no_cache_total:>18,} FLOPS')
print(f'  캐시 사용 : {with_cache_total:>18,} FLOPS')
print(f'  속도 향상 : {speedup:>18.1f}x')

# 메모리 분석
kv_cache_bytes = 2 * EXP_n_layers * EXP_max_context * EXP_n_kv_heads * EXP_d_head * EXP_bits
kv_cache_gb    = kv_cache_bytes / 1e9

print(f'\n{sep2}')
print(f'[KV 캐시 메모리 -- 최대 컨텍스트 {EXP_max_context:,} 토큰]')
print(sep2)
print(f'  KV 캐시 크기: {kv_cache_gb:.3f} GB')
print(f'  (= 2 x {EXP_n_layers}레이어 x {EXP_max_context:,}토큰 x '
      f'{EXP_n_kv_heads}KV헤드 x {EXP_d_head}차원 x {EXP_bits}바이트)')

# 컨텍스트 길이별 메모리 (간단 바 차트)
print(f'\n[다양한 컨텍스트 길이에서의 KV 캐시 메모리]')
print(f'  {"컨텍스트":>10} | {"메모리":>10} | 상대적 크기')
print('  ' + '-' * 50)
for ctx in [512, 1024, 2048, 4096, 8192, 32768, 128000]:
    size_gb = 2 * EXP_n_layers * ctx * EXP_n_kv_heads * EXP_d_head * EXP_bits / 1e9
    bar_len = max(1, int(size_gb * 20 / kv_cache_gb)) if kv_cache_gb > 0 else 1
    bar = '#' * min(bar_len, 30)
    marker = ' <- 현재 설정' if ctx == EXP_max_context else ''
    print(f'  {ctx:>10,} | {size_gb:>7.2f} GB  | {bar}{marker}')

print(f'\n실험 제안:')
print(f'  1) EXP_seq_len: 100 -> 1000 -> 10000 으로 바꾸면 속도 향상이 얼마나 달라지나요?')
_gqa_demo = max(2, EXP_n_heads // 4)  # n_heads가 작을 때 MQA(1)와 안 겹치도록 최소 2 보장
print(f'  2) EXP_n_kv_heads: {EXP_n_heads}(MHA) -> {_gqa_demo}(GQA) -> 1(MQA) 로 바꾸면 메모리가 얼마나 줄어드나요?')
print(f'  3) EXP_bits: 2.0(FP16) -> 0.5(INT4) 로 바꾸면 128K 컨텍스트 메모리가 얼마나 되나요?')
print(f'  4) EXP_n_layers: 12(GPT-2) -> 32(LLaMA-7B) -> 96(GPT-4 추정) 로 바꿔보세요.')

## 핵심 정리

### KV 캐시란?
자기회귀 생성 시, **한 번 계산한 Key(K)와 Value(V)를 메모리에 저장**하여  
다음 스텝에서 재사용하는 최적화 기법입니다.

### 왜 K, V만 캐싱하고 Q는 안 하나요?
- **Q (Query)**: 새로 생성되는 토큰에 대해서만 필요하고, 매 스텝 달라집니다
- **K, V**: 이전 모든 토큰에 대해 필요하지만, 이미 생성된 토큰의 값은 변하지 않습니다

### 효과 요약

| 항목 | 캐시 없음 | 캐시 사용 |
|------|---------|----------|
| 스텝 t에서 K,V *투영* 계산 횟수 | t번 (전체 재계산) | 1번 (새 토큰만) |
| 1000 토큰 기준, K,V 투영 계산 횟수 비율 | 기준 | 약 500배 적음 |
| 1000 토큰 기준, **전체 FLOPS** 속도 향상 | 기준 | 위 코드 셀에서 직접 측정한 값 참고 |

K,V 투영 계산만 보면 이론상 N배 가까이 줄어들지만, Attention Score 계산(Q·Kᵀ)은  
캐시 여부와 무관하게 시퀀스 길이에 비례해 항상 필요하므로 **전체 FLOPS 기준 속도 향상은 이보다 작습니다**.  
그래도 실무적으로는 충분히 의미 있는 수준의 가속입니다.

### 단점과 해결책
- **단점**: 시퀀스 길이에 비례해 메모리 사용량이 증가합니다
- **해결책**:
  - **GQA** (Grouped Query Attention): 여러 Q 헤드가 K, V를 공유 -> 메모리 절약
  - **MQA** (Multi-Query Attention): 극단적인 공유 -> 최대 메모리 절약
  - **양자화 (INT8/INT4)**: 낮은 정밀도로 저장 -> 2~4배 메모리 절약

### 실제 활용
모든 현대 LLM 추론 엔진(vLLM, TensorRT-LLM, TGI 등)은 KV 캐시를 기본으로 사용합니다.  
긴 문서 처리, 챗봇 멀티턴 대화, RAG 등 실제 서비스에서 필수적인 기술입니다.

---
*다음 주제 예고: **페이지드 어텐션(PagedAttention)** — KV 캐시를 OS의 가상 메모리처럼 관리하여  
더 많은 요청을 동시에 처리하는 기법 (vLLM의 핵심 아이디어)*